# Salary Prediction Demo

This notebook demonstrates the complete workflow for salary prediction using our machine learning pipeline.

## Contents
1. Setup and Data Generation
2. Data Exploration
3. Data Preprocessing
4. Model Training
5. Model Evaluation
6. Making Predictions
7. Visualization

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import DataPreprocessor
from src.training import ModelTrainer
from src.prediction import SalaryPredictor
from src.utils import (
    create_sample_data,
    print_data_summary,
    plot_feature_importance,
    plot_predictions_vs_actual,
    plot_residuals,
    load_config
)

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully!")

## 1. Setup and Data Generation

Let's create some sample data for demonstration purposes.

In [ ]:
# Generate sample data
df = create_sample_data(
    n_samples=1000,
    output_path='../data/raw/salary_data.csv',
    random_state=42
)

print(f"\nGenerated {len(df)} samples")
df.head()

## 2. Data Exploration

Let's explore the dataset to understand its structure and characteristics.

In [ ]:
# Print comprehensive data summary
print_data_summary(df)

In [ ]:
# Visualize salary distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df['salary'], bins=30, edgecolor='black')
plt.xlabel('Salary')
plt.ylabel('Frequency')
plt.title('Salary Distribution')

plt.subplot(1, 2, 2)
plt.boxplot(df['salary'])
plt.ylabel('Salary')
plt.title('Salary Box Plot')

plt.tight_layout()
plt.show()

In [ ]:
# Explore relationships between features and salary
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Feature Analysis', fontsize=16)

# Years of experience vs Salary
axes[0, 0].scatter(df['years_experience'], df['salary'], alpha=0.5)
axes[0, 0].set_xlabel('Years Experience')
axes[0, 0].set_ylabel('Salary')
axes[0, 0].set_title('Experience vs Salary')

# Salary by Education Level
df.boxplot(column='salary', by='education_level', ax=axes[0, 1])
axes[0, 1].set_title('Salary by Education')
axes[0, 1].set_xlabel('Education Level')

# Salary by Job Title
df.boxplot(column='salary', by='job_title', ax=axes[0, 2])
axes[0, 2].set_title('Salary by Job Title')
axes[0, 2].set_xlabel('Job Title')

# Salary by Location
df.boxplot(column='salary', by='location', ax=axes[1, 0])
axes[1, 0].set_title('Salary by Location')
axes[1, 0].set_xlabel('Location')

# Salary by Company Size
df.boxplot(column='salary', by='company_size', ax=axes[1, 1])
axes[1, 1].set_title('Salary by Company Size')
axes[1, 1].set_xlabel('Company Size')

# Salary by Industry
df.boxplot(column='salary', by='industry', ax=axes[1, 2])
axes[1, 2].set_title('Salary by Industry')
axes[1, 2].set_xlabel('Industry')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

Prepare the data for model training.

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor(random_state=42)

# Define categorical columns
categorical_columns = ['education_level', 'job_title', 'location', 'company_size', 'industry']

# Run preprocessing pipeline
X_train, X_test, y_train, y_test = preprocessor.preprocess_pipeline(
    filepath='../data/raw/salary_data.csv',
    target_column='salary',
    categorical_columns=categorical_columns,
    test_size=0.2,
    scale=True
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Feature names: {preprocessor.feature_names}")

## 4. Model Training

Train multiple models and compare their performance.

In [ ]:
# Initialize trainer
trainer = ModelTrainer(random_state=42)

# Compare multiple models
comparison_df = trainer.compare_models(
    X_train, y_train, X_test, y_test,
    model_types=['linear', 'ridge', 'random_forest', 'gradient_boosting']
)

comparison_df

## 5. Model Evaluation

Evaluate the best model in detail.

In [ ]:
# Train the best model (Random Forest)
model = trainer.train_model(
    X_train, y_train,
    model_type='random_forest',
    n_estimators=100,
    max_depth=20,
    random_state=42
)

# Evaluate
metrics = trainer.evaluate_model(X_test, y_test)

# Cross-validation
cv_results = trainer.cross_validate(X_train, y_train, cv=5)

In [ ]:
# Get feature importance
importance_df = trainer.get_feature_importance(preprocessor.feature_names)

# Plot feature importance
plot_feature_importance(importance_df, top_n=10)

In [ ]:
# Generate predictions for visualization
y_pred = model.predict(X_test)

# Plot predictions vs actual
plot_predictions_vs_actual(y_test, y_pred)

In [ ]:
# Plot residuals
plot_residuals(y_test, y_pred)

## 6. Making Predictions

Use the trained model to make predictions.

In [ ]:
# Save the model
trainer.save_model('../models/salary_prediction_model.pkl')

# Load the model using predictor
predictor = SalaryPredictor('../models/salary_prediction_model.pkl')

In [ ]:
# Make a single prediction
# Note: Features should be in the same order and encoding as training data
sample_features = pd.DataFrame([{
    'years_experience': 5,
    'education_level': 1,  # Master's
    'job_title': 2,        # Senior
    'location': 1,         # San Francisco
    'company_size': 2,     # Large
    'industry': 0          # Tech
}])

# Scale the features
sample_features_scaled = preprocessor.scaler.transform(sample_features)

# Predict
predicted_salary = predictor.predict(sample_features_scaled)[0]

print(f"\nPrediction for:")
print(f"  Years Experience: 5")
print(f"  Education: Master's")
print(f"  Job Title: Senior")
print(f"  Location: San Francisco")
print(f"  Company Size: Large")
print(f"  Industry: Tech")
print(f"\nPredicted Salary: ${predicted_salary:,.2f}")

## 7. Summary

In this notebook, we:
1. Generated sample salary data
2. Explored the data and visualized relationships
3. Preprocessed the data (cleaning, encoding, scaling)
4. Trained and compared multiple models
5. Evaluated model performance
6. Made predictions on new data

The Random Forest model showed the best performance with good accuracy in predicting salaries based on the given features.